# T-Lab 2026 — final V3 activation-steering experiment

One top-to-bottom Kaggle run builds the 200k clean-activation bank, trains the unconditional reconstruction control, and four intervention-aware ablations. The narrow question is whether `D(x,v,s)` improves on `D(x)`. After validation selection and protocol freeze it opens the new V3 holdout and one non-SAE sentiment confirmation. No prior output Dataset is required.


## Final V3 protocol map

0. Title / research question
1. Imports / config
2. Reproducibility
3. Load project
4. Load GPT-2 + SAE
5. Compatibility audit
6. Direction splits
7. Activation statistics / calibration
8. Steering definitions
9. Metric definitions
10. Baseline sanity tests
11. Harmfulness analysis
12. Denoiser architectures
13. Loss scale calibration
14. Training
15. Validation evaluation
16. Ablation analysis
17. Validation model selection
18. Freeze protocol
19. New untouched holdout evaluation
20. Generation
21. Independent semantic proxy and post-freeze cross-concept confirmation
22. Correction geometry
23. Hierarchical paired statistics
24. Figures
25. Final tables
26. Hypothesis-by-hypothesis conclusions
27. Limitations
28. Artifact manifest

The early cells execute the reusable V1 prerequisites. The final V3 call follows
the numbered order above inside `run_v3.py` and fails loudly at every gate.


## 1. Bootstrap


In [1]:
from pathlib import Path
import os, shutil, zipfile

working_root = Path('/kaggle/working/steering-denoiser')
input_root = Path('/kaggle/input')
required_source_files = ('model.py', 'steering.py', 'train.py', 'experiment.py')

def is_project_root(path):
    path = Path(path)
    return (path/'config.yaml').is_file() and (path/'requirements.txt').is_file() and all((path/'src'/name).is_file() for name in required_source_files)

# Preferred order: current checkout, previous working copy, unpacked Kaggle Dataset, ZIP fallback.
if is_project_root(Path.cwd()):
    project_root = Path.cwd()
elif is_project_root(working_root):
    project_root = working_root
else:
    unpacked = sorted({p.parent for p in input_root.rglob('config.yaml') if is_project_root(p.parent)})
    if unpacked:
        # Prefer a previous successful notebook output containing reusable V1 artifacts.
        def reusable_artifact_score(path):
            required = (
                'outputs/activations/stats.pt',
                'outputs/direction_split.json',
                'outputs/checkpoints/gaussian.pt',
                'outputs/checkpoints/sae_calibrated.pt',
                'outputs/checkpoints/fluency.pt',
            )
            return sum((path/item).is_file() for item in required)
        source_root = max(unpacked, key=lambda path: (reusable_artifact_score(path), str(path)))
        print('Selected input project:', source_root, 'reusable artifacts:', reusable_artifact_score(source_root), '/ 5')
        working_root.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_root, working_root, dirs_exist_ok=True)
        project_root = working_root
    else:
        archives = sorted(input_root.rglob('*.zip'))
        preferred = [p for p in archives if 'steering' in p.name.lower() or 't_lab' in p.name.lower() or 'tlab' in p.name.lower()]
        archive = preferred[0] if preferred else (archives[0] if archives else None)
        if archive is None:
            attached = sorted(str(p.relative_to(input_root)) for p in input_root.iterdir()) if input_root.exists() else []
            raise FileNotFoundError('Project source was not attached. In Kaggle click Add Input and attach the unpacked project Dataset (recommended) or a project ZIP. Visible /kaggle/input entries: ' + repr(attached))
        working_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as project_zip:
            project_zip.extractall(working_root)
        extracted = sorted({p.parent for p in working_root.rglob('config.yaml') if is_project_root(p.parent)})
        if len(extracted) != 1:
            raise RuntimeError(f'Expected exactly one project in {archive}, found {len(extracted)}: {extracted}')
        project_root = extracted[0]
os.chdir(project_root)
assert is_project_root(Path.cwd())
print('Project root:', Path.cwd())


Selected input project: /kaggle/input/datasets/leolazz/mech-tlab reusable artifacts: 0 / 5
Project root: /kaggle/working/steering-denoiser


## 2. Dependencies


In [2]:
import importlib.util, subprocess, sys
required = {'torch': 'torch', 'transformer_lens': 'transformer-lens',
    'sae_lens': 'sae-lens', 'transformers': 'transformers', 'datasets': 'datasets',
    'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib', 'yaml': 'pyyaml', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependency check complete')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.1/313.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.7/334.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.7/241.7 kB 17.3 MB/s eta 0:00:00
Dependency check complete


## 3. Imports, config, seeds, and run flags


In [3]:
import json
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset

from src.denoiser import ResidualDenoiser
from src.directions import (direction_split_hash, get_or_create_direction_split,
    load_direction_split, load_sae_directions, model_sae_compatibility_report,
    validate_direction_ids_for_usage)
from src.experiment import (cache_residual_activations, compact_activation_cache_summary,
    RuntimeProfile, evaluate_generation_methods, evaluate_token_level_methods,
    evaluate_token_level_methods_fast, freeze_test_config,
    get_or_create_text_bank, load_frozen_test_config, load_or_cache_residual_activations,
    positive_strength_concept_retention, regression_check_optimized_evaluation,
    score_direction_harmfulness_fast, split_activation_shards,
    summarize_concept_retention, tokenize_text_batches,
    tokenize_text_batches_with_oom_fallback)
from src.metrics import fit_natural_neighbor_index_from_shards, spearman_neighbor_correlations
from src.model import load_model, sanity_check_identity_intervention
from src.train import (CORRUPTION_SAE_CALIBRATED, CORRUPTION_SAE_HARMFUL,
    assert_calibrated_training_batch_math, calibrated_sampler_math_gate,
    correction_diagnostics, denoising_sanity_metrics, load_denoiser_checkpoint,
    make_calibrated_sae_corruption_sampler, make_fluency_sensitive_corruption_sampler,
    make_gaussian_corruption_sampler, prepare_training_corruption_batch,
    train_calibrated_sae_denoiser, train_fluency_sensitive_denoiser,
    train_gaussian_denoiser, validate_mixture_sampling)
from src.utils import (PIPELINE_VERSION, create_output_directories,
    denormalize_activations, file_fingerprint, generate_standard_figures, load_config,
    normalize_activations, run_pipeline_audit, seed_everything)

DEBUG = False
FORCE_REBUILD_ACTIVATIONS = True
FORCE_RETRAIN_GAUSSIAN = True
FORCE_RETRAIN_CALIBRATED = True
FORCE_RECOMPUTE_HARMFULNESS = True
FORCE_RETRAIN_FLUENCY = True
RUN_PROJECTED = True
RUN_INCREMENTAL = True
RUN_QUICK_VALIDATION = True
RUN_FULL_VALIDATION = True
RUN_NEIGHBOR_DIAGNOSTICS = True
RUN_GENERATION = False
RUN_TEST = False
EVAL_INTERVENTION_BATCH = 8
TOKEN_EVAL_PROMPT_BATCH_SIZE = 16
USE_INFERENCE_AUTOCAST = True
HARMFULNESS_NUM_DIRECTIONS = 100
HARMFULNESS_NUM_CONTEXTS = 32
HARMFULNESS_STRENGTHS = [0.25, 0.5]

config = load_config('config.yaml', debug=DEBUG)
assert config['pipeline_version'] == PIPELINE_VERSION
if DEBUG:
    EVAL_INTERVENTION_BATCH = config['evaluation']['intervention_batch_size']
    TOKEN_EVAL_PROMPT_BATCH_SIZE = config['evaluation']['token_eval_prompt_batch_size']
    USE_INFERENCE_AUTOCAST = config['evaluation']['use_inference_autocast']
    HARMFULNESS_NUM_DIRECTIONS = config['directions']['num_train']
    HARMFULNESS_NUM_CONTEXTS = config['damage_score']['max_contexts']
    HARMFULNESS_STRENGTHS = config['damage_score']['relative_strengths']
config['evaluation']['intervention_batch_size'] = EVAL_INTERVENTION_BATCH
config['evaluation']['token_eval_prompt_batch_size'] = TOKEN_EVAL_PROMPT_BATCH_SIZE
config['evaluation']['use_inference_autocast'] = USE_INFERENCE_AUTOCAST
config['damage_score']['max_contexts'] = HARMFULNESS_NUM_CONTEXTS
config['damage_score']['relative_strengths'] = HARMFULNESS_STRENGTHS
SAE_IDENTITY = f"{config['sae']['release']}:{config['sae']['sae_id']}"
if RUN_TEST:
    assert not DEBUG, 'Final test is forbidden in DEBUG mode.'
    assert Path('outputs/frozen_test_config.json').exists(), (
        'RUN_TEST=True requires a frozen config created by an earlier validation run.')
seed_everything(config['seed'], config['reproducibility']['deterministic_algorithms'])
create_output_directories('outputs')
runtime_profile = RuntimeProfile()
if not DEBUG:
    # A saved Kaggle version starts with a fresh /kaggle/working directory,
    # so it cannot require outputs/debug/audit.json from an earlier run.
    # The full run executes the same inline gates and the mandatory final audit.
    print('FULL run: prior DEBUG artifact is not required; inline gates remain enabled')
print({'DEBUG': DEBUG, 'pipeline_version': PIPELINE_VERSION, 'seed': config['seed']})


FULL run: prior DEBUG artifact is not required; inline gates remain enabled
{'DEBUG': False, 'pipeline_version': 'final_v2', 'seed': 42}


## 4. Load GPT-2 Small


In [4]:
model = load_model(config['model']['name'], config['model']['device'], config['model']['dtype'])
hook_name = config['model']['hook_name']
assert hook_name == 'blocks.6.hook_resid_pre'
if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
sanity_check_identity_intervention(model, hook_name)
print('Model identity-hook gate passed:', model.cfg.d_model, hook_name)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Model identity-hook gate passed: 768 blocks.6.hook_resid_pre


## 5. Load the matching SAE and normalized decoder directions


In [5]:
sae, all_directions = load_sae_directions(
    model.cfg.d_model, config['sae']['release'], config['sae']['sae_id'],
    device=config['sae']['device'], dtype=config['sae']['dtype'])
assert all_directions.shape[1] == model.cfg.d_model
print('SAE directions:', tuple(all_directions.shape))


cfg.json: 0.00B [00:00, ?B/s]

blocks.6.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.6.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

SAE directions: (24576, 768)


/usr/local/lib/python3.12/dist-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


## 6. Verify model and SAE preprocessing compatibility


In [6]:
compatibility = model_sae_compatibility_report(model, sae, hook_name)
print(json.dumps(compatibility, indent=2, default=str))
assert compatibility['compatible']


{
  "hook_name": "blocks.6.hook_resid_pre",
  "sae_hook_name": "blocks.6.hook_resid_pre",
  "hook_matches": true,
  "model_d_model": 768,
  "sae_d_in": 768,
  "dimensions_match": true,
  "expected_model_kwargs": {
    "center_writing_weights": true
  },
  "compared_model_cfg": {
    "center_writing_weights": {
      "expected": true,
      "actual": true,
      "matches": true,
      "source": "load_kwargs"
    }
  },
  "unrecognized_expected_kwargs": {},
  "compatible": true
}


## 7. Direction split and leakage gates


In [7]:
direction_cfg = config['directions']
train_ids, val_ids, test_ids = get_or_create_direction_split(
    all_directions.shape[0], direction_cfg['num_train'], direction_cfg['num_validation'],
    direction_cfg['num_test'], config['seed'], direction_cfg['split_path'])
direction_split = load_direction_split(direction_cfg['split_path'])
validate_direction_ids_for_usage(train_ids, direction_split, 'training', require_complete_split=True)
validate_direction_ids_for_usage(val_ids, direction_split, 'preliminary_evaluation', require_complete_split=True)
print({'train': len(train_ids), 'val': len(val_ids), 'test': len(test_ids),
       'split_hash': direction_split_hash(direction_split)[:16]})


{'train': 100, 'val': 20, 'test': 20, 'split_hash': '2e8947a1e3a3611a'}


## 8. Load or build the activation bank


In [8]:
data_cfg = config['data']
activation_dir = Path(data_cfg['activation_output_dir'])
cache_exists = activation_dir.joinpath('stats.pt').exists()
dataset_stream = None
if FORCE_REBUILD_ACTIVATIONS or not cache_exists:
    dataset_stream = load_dataset(data_cfg['dataset_name'], split=data_cfg['split'], streaming=True)
if FORCE_REBUILD_ACTIVATIONS:
    stats = cache_residual_activations(dataset_stream, model, hook_name,
        data_cfg['target_num_activations'], activation_dir, data_cfg['text_column'],
        data_cfg['activation_batch_size'], data_cfg['max_length'],
        data_cfg['activation_shard_size'], data_cfg['activation_storage_dtype'], overwrite=True,
        dataset_name=data_cfg['dataset_name'])
else:
    stats = load_or_cache_residual_activations(dataset_stream if dataset_stream is not None else [], model, hook_name,
        data_cfg['target_num_activations'], activation_dir,
        text_column=data_cfg['text_column'], batch_size=data_cfg['activation_batch_size'],
        max_length=data_cfg['max_length'], shard_size=data_cfg['activation_shard_size'],
        storage_dtype=data_cfg['activation_storage_dtype'], dataset_name=data_cfg['dataset_name'])
print(compact_activation_cache_summary(stats))
train_activations, validation_activations = split_activation_shards(
    activation_dir, config['training']['validation_fraction'], config['seed'])


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Caching activations:   0%|          | 0/200000 [00:00<?, ?tok/s]

{'num_tokens': 200000, 'd_model': 768, 'mean_activation_norm': 90.69428253173828, 'median_activation_norm': 78.6480712890625, 'p95_activation_norm': 93.69611358642578, 'p99_activation_norm': 100.81643676757812, 'coordinate_std_min': 1.5921669006347656, 'coordinate_std_median': 1.9450443983078003, 'coordinate_std_max': 181.31399536132812, 'num_shards': 8}


## 9. Normalization sanity


In [9]:
normalization_iterator = iter(train_activations)
sample_raw = torch.stack([next(normalization_iterator) for _ in range(2)]).float()
mean_cpu, std_cpu = stats['mean'].float(), stats['std'].float()
raw_before = sample_raw.clone()
z = normalize_activations(sample_raw, mean_cpu, std_cpu)
roundtrip = denormalize_activations(z, mean_cpu, std_cpu)
torch.testing.assert_close(roundtrip, sample_raw, rtol=2e-5, atol=2e-5)
torch.testing.assert_close(sample_raw, raw_before, rtol=0, atol=0)
print('Normalization round-trip and no-mutation gates passed')


Normalization round-trip and no-mutation gates passed


## 10. Calibrated corruption mathematical gate


In [10]:
noise_cfg = config['noise']
calibrated_sampler = make_calibrated_sae_corruption_sampler(
    all_directions, direction_split, stats,
    noise_cfg['calibrated_magnitude_min'], noise_cfg['calibrated_magnitude_max'],
    noise_cfg['calibrated_gaussian_probability'], noise_cfg['gaussian_sigma_min'],
    noise_cfg['gaussian_sigma_max'])
calibrated_gate = calibrated_sampler_math_gate(
    calibrated_sampler.samplers[0], model.cfg.d_model,
    magnitudes=[noise_cfg['calibrated_magnitude_min'], noise_cfg['calibrated_magnitude_max']])
print(calibrated_gate.to_string(index=False))


 magnitude  actual_delta_norm  expected_delta_norm  actual_mse  expected_mse  relative_error
       3.8           3.800000                  3.8    0.018802      0.018802    0.000000e+00
      19.1          19.100002                 19.1    0.475013      0.475013    1.254800e-07


## 11. Exact real-training-batch sampler gate


In [11]:
gate_rows = []
gate_iterator = iter(train_activations)
for _ in range(min(config['training']['batch_size'], 64)):
    gate_rows.append(next(gate_iterator))
clean_gate = torch.stack(gate_rows).to(next(model.parameters()).device, dtype=next(model.parameters()).dtype)
mean_device = mean_cpu.to(clean_gate)
std_device = std_cpu.to(clean_gate)
clean_z_gate, gate_corruption = prepare_training_corruption_batch(
    clean_gate, calibrated_sampler, mean_device, std_device,
    torch.Generator(device=clean_gate.device).manual_seed(config['seed']),
    config['training']['clean_identity_probability'])
training_math = assert_calibrated_training_batch_math(clean_z_gate, gate_corruption)
mixture_gate = validate_mixture_sampling(
    calibrated_sampler, config['training']['clean_identity_probability'], seed=config['seed'])
print('REAL batch math:', training_math)
print('Mixture frequencies:', mixture_gate)


REAL batch math: {'num_sae': 52.0, 'mean_actual_norm': 12.225171089172363, 'mean_expected_norm': 12.225171089172363, 'mean_actual_mse': 0.22004425525665283, 'mean_expected_mse': 0.22004425525665283}
Mixture frequencies: {'requested': {'identity': 0.1, 'sae_calibrated': 0.765, 'gaussian': 0.135}, 'empirical': {'identity': 0.09600000083446503, 'sae_calibrated': 0.7667999863624573, 'gaussian': 0.1371999979019165}, 'num_samples': 10000}


## 12. Raw baseline sanity


In [12]:
context_count = max(config['damage_score']['max_contexts'], config['evaluation']['num_prompts'])
context_bank_path = activation_dir.parent / 'context_bank.json'
if not context_bank_path.exists() and dataset_stream is None:
    dataset_stream = load_dataset(data_cfg['dataset_name'], split=data_cfg['split'], streaming=True)
context_texts = get_or_create_text_bank(dataset_stream, context_count, context_bank_path, data_cfg['text_column'])
full_texts = context_texts[:config['evaluation']['num_prompts']]
full_batches, accepted_prompt_batch = tokenize_text_batches_with_oom_fallback(
    model, full_texts, TOKEN_EVAL_PROMPT_BATCH_SIZE, data_cfg['max_length'],
    use_inference_autocast=USE_INFERENCE_AUTOCAST, profiler=runtime_profile)
quick_texts = full_texts[:config['evaluation']['quick_num_prompts']]
quick_batches = tokenize_text_batches(model, quick_texts, accepted_prompt_batch, data_cfg['max_length'])
smoke_batches = full_batches
raw_smoke = evaluate_token_level_methods(model, smoke_batches[:1], all_directions,
    direction_split, val_ids[:1], [0.0, 0.5], hook_name, ['raw'],
    evaluation_split='val', sae=sae)
print(raw_smoke[['method','strength','kl','delta_nll','activation_norm_ratio','target_sae_activation']])


  method  strength        kl  delta_nll  activation_norm_ratio  \
0    raw       0.0  0.000000   0.000000               1.000000   
1    raw       0.5  3.503417   3.589281               1.117252   

   target_sae_activation  
0               0.000993  
1              28.862743  


## 13. Train or reuse the Gaussian baseline


In [13]:
def new_denoiser():
    dcfg = config['denoiser']
    return ResidualDenoiser(model.cfg.d_model, dcfg['hidden_dim'],
        dcfg['condition_on_noise'], dcfg['conditioning_hidden_dim']).to(
        device=next(model.parameters()).device, dtype=next(model.parameters()).dtype)

def load_if_compatible(path, mode):
    try:
        return load_denoiser_checkpoint(path, config['model']['device'], config['model']['dtype'],
            expected_hook_name=hook_name, expected_model_name=config['model']['name'],
            expected_corruption_mode=mode,
            expected_direction_split_hash=direction_split_hash(direction_split))
    except (FileNotFoundError, AssertionError, RuntimeError, KeyError) as error:
        print(f'Retraining {mode}: {error}')
        return None

paths = config['training']['checkpoint_paths']
gaussian_loaded = None if FORCE_RETRAIN_GAUSSIAN else load_if_compatible(paths['gaussian'], 'gaussian')
if gaussian_loaded is None:
    train_gaussian_denoiser(new_denoiser(), train_activations, validation_activations, config, stats)
    gaussian_loaded = load_if_compatible(paths['gaussian'], 'gaussian')
assert gaussian_loaded is not None
gaussian_denoiser, gaussian_checkpoint = gaussian_loaded


Denoiser epoch 1/10:   0%|          | 0/2735 [00:00<?, ?it/s]

step=1 train_loss=1.027355 clean_identity_loss=0.000000
step=50 train_loss=1.266867 clean_identity_loss=0.000082
step=100 train_loss=1.155030 clean_identity_loss=0.000148
step=150 train_loss=0.985029 clean_identity_loss=0.000296
step=200 train_loss=1.162561 clean_identity_loss=0.000794
step=250 train_loss=1.215444 clean_identity_loss=0.001392
step=300 train_loss=1.281209 clean_identity_loss=0.003340
step=350 train_loss=1.207872 clean_identity_loss=0.004217
step=400 train_loss=0.991351 clean_identity_loss=0.005415
step=450 train_loss=1.260875 clean_identity_loss=0.005723
step=500 train_loss=1.171946 clean_identity_loss=0.007611
step=550 train_loss=1.188329 clean_identity_loss=0.007918
step=600 train_loss=1.076218 clean_identity_loss=0.005486
step=650 train_loss=1.276125 clean_identity_loss=0.006319
step=700 train_loss=1.008193 clean_identity_loss=0.004755
step=750 train_loss=1.153565 clean_identity_loss=0.003364
step=800 train_loss=1.112683 clean_identity_loss=0.003931
step=850 train_lo

Denoiser epoch 2/10:   0%|          | 0/2735 [00:00<?, ?it/s]

step=2750 train_loss=0.770925 clean_identity_loss=0.001711
step=2800 train_loss=0.932414 clean_identity_loss=0.001997
step=2850 train_loss=0.806043 clean_identity_loss=0.002396
step=2900 train_loss=0.625554 clean_identity_loss=0.001574
step=2950 train_loss=0.768314 clean_identity_loss=0.001940
step=3000 train_loss=0.733126 clean_identity_loss=0.001833
epoch=2 step=3000 val_loss=0.861949 val_clean_identity_loss=0.001773


## 14. Train corrected calibrated SAE baseline


In [14]:
calibrated_loaded = None if FORCE_RETRAIN_CALIBRATED else load_if_compatible(paths['sae_calibrated'], 'sae_calibrated')
if calibrated_loaded is None:
    train_calibrated_sae_denoiser(new_denoiser(), train_activations, validation_activations,
        all_directions, direction_split, config, stats,
        noise_cfg['calibrated_magnitude_min'], noise_cfg['calibrated_magnitude_max'])
    calibrated_loaded = load_if_compatible(paths['sae_calibrated'], 'sae_calibrated')
assert calibrated_loaded is not None
calibrated_denoiser, calibrated_checkpoint = calibrated_loaded
assert calibrated_checkpoint['real_training_math_verified']
print(calibrated_checkpoint['real_training_math_summary'])


Corruption mixture gate: {'requested': {'identity': 0.1, 'sae_calibrated': 0.765, 'gaussian': 0.135}, 'empirical': {'identity': 0.09600000083446503, 'sae_calibrated': 0.7667999863624573, 'gaussian': 0.1371999979019165}, 'num_samples': 10000}


Denoiser epoch 1/10:   0%|          | 0/2735 [00:00<?, ?it/s]

REAL training-batch calibrated math: {'num_sae': 52.0, 'mean_actual_norm': 12.225171089172363, 'mean_expected_norm': 12.225171089172363, 'mean_actual_mse': 0.22004425525665283, 'mean_expected_mse': 0.22004425525665283}
step=1 train_loss=0.262631 clean_identity_loss=0.000000
step=50 train_loss=0.265097 clean_identity_loss=0.000034
step=100 train_loss=0.293469 clean_identity_loss=0.000079
step=150 train_loss=0.314875 clean_identity_loss=0.000241
step=200 train_loss=0.262165 clean_identity_loss=0.001455
step=250 train_loss=0.377728 clean_identity_loss=0.003198
step=300 train_loss=0.151641 clean_identity_loss=0.002847
step=350 train_loss=0.249153 clean_identity_loss=0.005146
step=400 train_loss=0.156010 clean_identity_loss=0.004356
step=450 train_loss=0.187920 clean_identity_loss=0.003673
step=500 train_loss=0.285367 clean_identity_loss=0.006230
step=550 train_loss=0.211624 clean_identity_loss=0.007183
step=600 train_loss=0.335524 clean_identity_loss=0.010144
step=650 train_loss=0.292809 c

Denoiser epoch 2/10:   0%|          | 0/2735 [00:00<?, ?it/s]

step=2750 train_loss=0.152354 clean_identity_loss=0.001633
step=2800 train_loss=0.271085 clean_identity_loss=0.001926
step=2850 train_loss=0.034294 clean_identity_loss=0.001537
step=2900 train_loss=0.206781 clean_identity_loss=0.001778
step=2950 train_loss=0.225942 clean_identity_loss=0.001863
step=3000 train_loss=0.045431 clean_identity_loss=0.001458
epoch=2 step=3000 val_loss=0.215859 val_clean_identity_loss=0.002207
{'num_sae': 56.0, 'mean_actual_norm': 12.430500030517578, 'mean_expected_norm': 12.430500030517578, 'mean_actual_mse': 0.22848279774188995, 'mean_expected_mse': 0.22848279774188995}


## 15. Reconstruction and correction diagnostics


In [15]:
gaussian_reconstruction = denoising_sanity_metrics(gaussian_denoiser, clean_gate,
    make_gaussian_corruption_sampler(config), stats, config['seed'])
calibrated_reconstruction = denoising_sanity_metrics(calibrated_denoiser, clean_gate,
    calibrated_sampler, stats, config['seed'])
print({'gaussian': gaussian_reconstruction, 'sae_calibrated': calibrated_reconstruction})


{'gaussian': {'corrupted_mse': 1.1241488456726074, 'denoised_mse': 0.6462905406951904, 'improves': True, 'mean_corruption_norm': 24.158523559570312, 'mean_correction_cosine': 0.426531046628952}, 'sae_calibrated': {'corrupted_mse': 0.3227461278438568, 'denoised_mse': 0.13275590538978577, 'improves': True, 'mean_corruption_norm': 13.604093551635742, 'mean_correction_cosine': 0.806336522102356}}


## 16. Validation baseline comparison


In [16]:
denoisers = {'gaussian': gaussian_denoiser, 'sae_calibrated': calibrated_denoiser}
normalizations = {'gaussian': gaussian_checkpoint['normalization'],
                  'sae_calibrated': calibrated_checkpoint['normalization']}
baseline_methods = ['raw', 'norm_preserving', 'gaussian_denoiser', 'sae_calibrated']
validation_dir = Path('outputs/debug/validation' if DEBUG else 'outputs/validation')
validation_dir.mkdir(parents=True, exist_ok=True)
analysis_dir = Path('outputs/debug/analysis' if DEBUG else 'outputs/analysis')
analysis_dir.mkdir(parents=True, exist_ok=True)
checkpoint_fingerprints = {name: file_fingerprint(path) for name, path in paths.items()
    if name in {'gaussian', 'sae_calibrated'} and Path(path).exists()}
regression_batches = tokenize_text_batches(model, quick_texts[:2], 1, data_cfg['max_length'])
regression = regression_check_optimized_evaluation(model, regression_batches, all_directions,
    direction_split, val_ids[:2], [0.0, 0.5], hook_name, ['raw', 'sae_calibrated'],
    denoisers, normalizations, config['evaluation'], sae,
    path=analysis_dir/'optimized_eval_regression.csv')
print('Optimized/reference gate passed; max absolute differences:')
print(regression.filter(like='_abs_diff').max().to_dict())
baseline_results = pd.DataFrame()
if RUN_QUICK_VALIDATION:
    baseline_results = evaluate_token_level_methods_fast(model, quick_batches, all_directions,
        direction_split, val_ids[:config['evaluation']['num_validation_directions']],
        config['evaluation']['quick_strengths'], hook_name, baseline_methods,
        denoisers, normalizations, config['evaluation'], 'val', sae,
        intervention_batch_size=EVAL_INTERVENTION_BATCH,
        use_inference_autocast=USE_INFERENCE_AUTOCAST,
        partial_jsonl_path=validation_dir/'baseline_quick.partial.jsonl',
        checkpoint_fingerprints=checkpoint_fingerprints, sae_identity=SAE_IDENTITY,
        profiler=runtime_profile)
    baseline_results.to_csv(validation_dir/'baseline_token_metrics.csv', index=False)
    print(baseline_results.groupby(['method','strength'])[['kl','delta_nll']].mean())
runtime_profile.save(analysis_dir/'runtime_profile.csv')


{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'raw', 'completed_configurations': 4, 'seconds': 0.12, 'configurations_per_second': 33.2}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'sae_calibrated', 'completed_configurations': 4, 'seconds': 0.12, 'configurations_per_second': 32.404}
{'stage': 'token_evaluation', 'prompt_batch': 1, 'method': 'raw', 'completed_configurations': 4, 'seconds': 0.12, 'configurations_per_second': 32.692}
{'stage': 'token_evaluation', 'prompt_batch': 1, 'method': 'sae_calibrated', 'completed_configurations': 4, 'seconds': 0.12, 'configurations_per_second': 32.033}
                 stage  seconds  clean_forwards  modified_forwards  configurations  examples  configurations_per_second
SAE feature extraction 0.049675               0                  0               0         0                   0.000000
         clean forward 0.050199               2                  0               0         2                   0.000000
dataframe constr

PosixPath('outputs/analysis/runtime_profile.csv')

## 17. Train-direction harmfulness scoring


In [17]:
score_cfg = config['damage_score']
score_path = Path(score_cfg['output_path'])
assert HARMFULNESS_NUM_DIRECTIONS == len(train_ids), (
    'Current leakage-safe scorer covers the complete TRAIN split; configure the split itself.')
scoring_batches = tokenize_text_batches(model, context_texts[:HARMFULNESS_NUM_CONTEXTS],
    score_cfg['batch_size'], data_cfg['max_length'])
score_payload = score_direction_harmfulness_fast(model, scoring_batches, all_directions,
    direction_split, HARMFULNESS_STRENGTHS, hook_name,
    HARMFULNESS_NUM_CONTEXTS, score_cfg['direction_batch_size'], score_path,
    score_cfg['csv_path'], score_cfg['histogram_path'],
    use_inference_autocast=USE_INFERENCE_AUTOCAST,
    partial_path=score_path.with_name(score_path.stem+'_partial.pt'),
    profiler=runtime_profile, force_recompute=FORCE_RECOMPUTE_HARMFULNESS,
    sae_identity=SAE_IDENTITY)
harmfulness_frame = pd.DataFrame({'direction_id': score_payload['direction_ids'].tolist(),
    'mean_kl': score_payload['mean_kl'].tolist(),
    'std_kl': score_payload['std_kl'].tolist(),
    'mean_delta_nll': score_payload['mean_delta_nll'].tolist()})
harmfulness_frame.to_csv(analysis_dir/'harmfulness.csv', index=False)
print({'directions_scored': len(score_payload['direction_ids']),
       'mean_KL': score_payload['mean_kl'].mean().item()})
runtime_profile.save(analysis_dir/'runtime_profile.csv')


Scoring train directions:   0%|          | 0/2 [00:00<?, ?batch/s]

Top 20 most damaging training directions:
 direction_id   mean_kl   std_kl  mean_delta_nll  kl_r_0.25  delta_nll_r_0.25  kl_r_0.5  delta_nll_r_0.5
        24507 11.036788 6.615030       11.102827   5.912461          5.957394 16.161115        16.248260
         8963 10.619827 5.114977       10.677716   7.240848          7.279049 13.998806        14.076383
        15093  9.039368 4.754777        9.088215   5.874061          5.899924 12.204674        12.276506
        19714  8.982188 6.912075        9.032975   4.358717          4.391942 13.605658        13.674009
         9911  8.673218 4.920855        8.717162   5.188565          5.216030 12.157871        12.218294
        19507  8.622148 4.683352        8.671780   5.393082          5.431024 11.851214        11.912535
        21739  8.272064 4.562702        8.347372   5.131154          5.192536 11.412975        11.502208
         6132  7.981221 4.835934        8.071549   4.333653          4.401192 11.628789        11.741905
         1697

PosixPath('outputs/analysis/runtime_profile.csv')

## 18. Harmfulness sampler diagnostics


In [18]:
fluency_sampler = make_fluency_sensitive_corruption_sampler(
    all_directions, direction_split, config, score_path, stats)
fluency_mixture_gate = validate_mixture_sampling(
    fluency_sampler, config['training']['clean_identity_probability'], seed=config['seed'])
print(fluency_mixture_gate)


{'requested': {'identity': 0.1, 'sae_harmful': 0.54, 'sae_calibrated': 0.225, 'gaussian': 0.135}, 'empirical': {'identity': 0.09600000083446503, 'sae_harmful': 0.5370000004768372, 'sae_calibrated': 0.2298000007867813, 'gaussian': 0.1371999979019165}, 'num_samples': 10000}


## 19. Train fluency-sensitive denoiser


In [19]:
fluency_loaded = None if FORCE_RETRAIN_FLUENCY else load_if_compatible(paths['fluency'], 'fluency_sensitive')
if fluency_loaded is None:
    train_fluency_sensitive_denoiser(new_denoiser(), train_activations,
        validation_activations, all_directions, direction_split, config, score_path, stats)
    fluency_loaded = load_if_compatible(paths['fluency'], 'fluency_sensitive')
assert fluency_loaded is not None
fluency_denoiser, fluency_checkpoint = fluency_loaded
assert fluency_checkpoint['real_training_math_verified']
denoisers['fluency'] = fluency_denoiser
normalizations['fluency'] = fluency_checkpoint['normalization']


Corruption mixture gate: {'requested': {'identity': 0.1, 'sae_harmful': 0.54, 'sae_calibrated': 0.225, 'gaussian': 0.135}, 'empirical': {'identity': 0.09600000083446503, 'sae_harmful': 0.5370000004768372, 'sae_calibrated': 0.2298000007867813, 'gaussian': 0.1371999979019165}, 'num_samples': 10000}


Denoiser epoch 1/10:   0%|          | 0/2735 [00:00<?, ?it/s]

REAL training-batch calibrated math: {'num_sae': 49.0, 'mean_actual_norm': 11.975212097167969, 'mean_expected_norm': 11.975212097167969, 'mean_actual_mse': 0.21435999870300293, 'mean_expected_mse': 0.21435998380184174}
step=1 train_loss=0.207962 clean_identity_loss=0.000000
step=50 train_loss=0.287138 clean_identity_loss=0.000033
step=100 train_loss=0.324685 clean_identity_loss=0.000058
step=150 train_loss=0.183558 clean_identity_loss=0.000566
step=200 train_loss=0.341631 clean_identity_loss=0.000978
step=250 train_loss=0.198603 clean_identity_loss=0.002629
step=300 train_loss=0.217849 clean_identity_loss=0.003436
step=350 train_loss=0.355154 clean_identity_loss=0.003208
step=400 train_loss=0.314590 clean_identity_loss=0.003506
step=450 train_loss=0.219319 clean_identity_loss=0.002795
step=500 train_loss=0.127586 clean_identity_loss=0.005307
step=550 train_loss=0.224679 clean_identity_loss=0.005132
step=600 train_loss=0.195259 clean_identity_loss=0.007498
step=650 train_loss=0.139036 c

Denoiser epoch 2/10:   0%|          | 0/2735 [00:00<?, ?it/s]

step=2750 train_loss=0.195372 clean_identity_loss=0.001710
step=2800 train_loss=0.196048 clean_identity_loss=0.001948
step=2850 train_loss=0.204016 clean_identity_loss=0.001408
step=2900 train_loss=0.108317 clean_identity_loss=0.001349
step=2950 train_loss=0.230155 clean_identity_loss=0.001440
step=3000 train_loss=0.190309 clean_identity_loss=0.001639
epoch=2 step=3000 val_loss=0.195932 val_clean_identity_loss=0.001949


## 20. Main validation comparison


In [20]:
main_methods = baseline_methods + ['fluency_denoiser']
if RUN_PROJECTED:
    main_methods.append('projected_fluency_denoiser')
if RUN_INCREMENTAL:
    main_methods.append('incremental_fluency')
checkpoint_fingerprints['fluency'] = file_fingerprint(paths['fluency'])
selected_val_ids = val_ids[:config['evaluation']['num_validation_directions']]
quick_main_results = pd.DataFrame()
if RUN_QUICK_VALIDATION:
    quick_main_results = evaluate_token_level_methods_fast(model, quick_batches, all_directions,
        direction_split, selected_val_ids, config['evaluation']['quick_strengths'],
        hook_name, main_methods, denoisers, normalizations, config['evaluation'], 'val', sae,
        intervention_batch_size=EVAL_INTERVENTION_BATCH,
        use_inference_autocast=USE_INFERENCE_AUTOCAST,
        partial_jsonl_path=validation_dir/'main_quick.partial.jsonl',
        checkpoint_fingerprints=checkpoint_fingerprints, sae_identity=SAE_IDENTITY,
        profiler=runtime_profile)
    quick_main_results.to_csv(validation_dir/'quick_results.csv', index=False)
validation_results = quick_main_results
if RUN_FULL_VALIDATION:
    validation_results = evaluate_token_level_methods_fast(model, full_batches, all_directions,
        direction_split, selected_val_ids, config['evaluation']['full_strengths'],
        hook_name, main_methods, denoisers, normalizations, config['evaluation'], 'val', sae,
        intervention_batch_size=EVAL_INTERVENTION_BATCH,
        use_inference_autocast=USE_INFERENCE_AUTOCAST,
        partial_jsonl_path=validation_dir/'main_results.partial.jsonl',
        checkpoint_fingerprints=checkpoint_fingerprints, sae_identity=SAE_IDENTITY,
        profiler=runtime_profile)
if validation_results.empty and RUN_TEST:
    saved_validation = validation_dir/'main_results.csv'
    assert saved_validation.exists(), 'Final test requires saved validation/main_results.csv.'
    validation_results = pd.read_csv(saved_validation)
assert not validation_results.empty, (
    'Enable validation or provide saved main_results.csv for final TEST.')
validation_results.to_csv(validation_dir/'main_token_metrics.csv', index=False)
validation_results.to_csv(validation_dir/'main_results.csv', index=False)
retention = positive_strength_concept_retention(validation_results)
retention_summary = summarize_concept_retention(retention)
print(retention_summary)
runtime_profile.save(analysis_dir/'runtime_profile.csv')


{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'raw', 'completed_configurations': 12, 'seconds': 1.8, 'configurations_per_second': 6.685}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'norm_preserving', 'completed_configurations': 12, 'seconds': 1.77, 'configurations_per_second': 6.794}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'gaussian_denoiser', 'completed_configurations': 12, 'seconds': 1.81, 'configurations_per_second': 6.615}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'sae_calibrated', 'completed_configurations': 12, 'seconds': 1.83, 'configurations_per_second': 6.542}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.84, 'configurations_per_second': 6.524}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.86, 'configurations_per_second': 6.441}
{'stage': 'token_evalu

PosixPath('outputs/analysis/runtime_profile.csv')

## 21. Projected correction


In [21]:
if RUN_PROJECTED:
    projected_tuning_rows = []
    if RUN_QUICK_VALIDATION:
        for beta in config['evaluation']['projected_beta_candidates']:
            tuning_config = dict(config['evaluation']); tuning_config['projected_beta'] = beta
            candidate = evaluate_token_level_methods_fast(model, quick_batches, all_directions,
                direction_split, selected_val_ids, config['evaluation']['quick_strengths'],
                hook_name, ['projected_fluency_denoiser'], denoisers, normalizations,
                tuning_config, 'val', sae, intervention_batch_size=EVAL_INTERVENTION_BATCH,
                use_inference_autocast=USE_INFERENCE_AUTOCAST,
                partial_jsonl_path=analysis_dir/f'projected_beta_{beta}.partial.jsonl',
                checkpoint_fingerprints={'fluency': checkpoint_fingerprints['fluency']},
                sae_identity=SAE_IDENTITY,
                profiler=runtime_profile)
            candidate['projected_beta'] = beta; projected_tuning_rows.append(candidate)
        pd.concat(projected_tuning_rows, ignore_index=True).to_csv(
            analysis_dir/'projected_tuning.csv', index=False)
    projected_summary = validation_results[validation_results.method == 'projected_fluency_denoiser'].groupby('strength')[['kl','delta_nll','concept_score']].mean()
    print(projected_summary)
else:
    print('Projected correction disabled')


{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.91, 'configurations_per_second': 6.282}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.91, 'configurations_per_second': 6.298}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.91, 'configurations_per_second': 6.275}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 12, 'seconds': 1.91, 'configurations_per_second': 6.289}
                 kl  delta_nll  concept_score
strength                                     
0.00       0.004656   0.006439       0.004197
0.10       0.301434   0.301380       1.514734
0.25       2.962483   2.990113      12.203160
0.50       7.896000   7.968256      36.712706
1.00      10.699063  10.7883

## 22. Incremental steering


In [22]:
if RUN_INCREMENTAL:
    incremental_tuning_rows = []
    if RUN_QUICK_VALIDATION:
        for n_steps in config['evaluation']['incremental_step_candidates']:
            tuning_config = dict(config['evaluation']); tuning_config['incremental_steps'] = n_steps
            candidate = evaluate_token_level_methods_fast(model, quick_batches, all_directions,
                direction_split, selected_val_ids, config['evaluation']['quick_strengths'],
                hook_name, ['incremental_fluency'], denoisers, normalizations,
                tuning_config, 'val', sae, intervention_batch_size=EVAL_INTERVENTION_BATCH,
                use_inference_autocast=USE_INFERENCE_AUTOCAST,
                partial_jsonl_path=analysis_dir/f'incremental_steps_{n_steps}.partial.jsonl',
                checkpoint_fingerprints={'fluency': checkpoint_fingerprints['fluency']},
                sae_identity=SAE_IDENTITY,
                profiler=runtime_profile)
            candidate['n_steps'] = n_steps; incremental_tuning_rows.append(candidate)
        pd.concat(incremental_tuning_rows, ignore_index=True).to_csv(
            analysis_dir/'incremental_tuning.csv', index=False)
    incremental_summary = validation_results[validation_results.method == 'incremental_fluency'].groupby('strength')[['kl','delta_nll','concept_score']].mean()
    print('n_steps =', config['evaluation']['incremental_steps'])
    print(incremental_summary)
else:
    print('Incremental steering disabled')


{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'incremental_fluency', 'completed_configurations': 12, 'seconds': 1.9, 'configurations_per_second': 6.305}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'incremental_fluency', 'completed_configurations': 12, 'seconds': 1.92, 'configurations_per_second': 6.253}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'incremental_fluency', 'completed_configurations': 12, 'seconds': 1.94, 'configurations_per_second': 6.185}
n_steps = 4
                 kl  delta_nll  concept_score
strength                                     
0.00       0.079295   0.084886       0.012661
0.10       1.897540   1.953312       0.908742
0.25       3.531253   3.589412       5.597713
0.50       7.238510   7.306737      32.236898
1.00      10.788452  10.878285      80.142893
2.00      15.245954  15.336511     208.312334


## 23. Natural-neighbor diagnostics


In [23]:
neighbor_results = None
if RUN_NEIGHBOR_DIAGNOSTICS:
    neighbor_cfg = config['natural_neighbors']
    neighbor_index = fit_natural_neighbor_index_from_shards(activation_dir,
        neighbor_cfg['n_components'], neighbor_cfg['k'], neighbor_cfg['max_fit_samples'],
        config['seed'], hook_name)
    neighbor_index.save(neighbor_cfg['index_path'])
    neighbor_results = evaluate_token_level_methods_fast(model, quick_batches[:1], all_directions,
        direction_split, selected_val_ids[:1], config['evaluation']['quick_strengths'], hook_name,
        main_methods, denoisers, normalizations, config['evaluation'], 'val', sae,
        neighbor_index=neighbor_index, intervention_batch_size=EVAL_INTERVENTION_BATCH,
        use_inference_autocast=USE_INFERENCE_AUTOCAST,
        partial_jsonl_path=analysis_dir/'neighbor_metrics.partial.jsonl',
        checkpoint_fingerprints=checkpoint_fingerprints, sae_identity=SAE_IDENTITY,
        profiler=runtime_profile)
    neighbor_results.to_csv(validation_dir/'neighbor_diagnostics.csv', index=False)
    neighbor_results.to_csv(analysis_dir/'neighbor_metrics.csv', index=False)
    print(spearman_neighbor_correlations(neighbor_results))


{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'raw', 'completed_configurations': 3, 'seconds': 1.29, 'configurations_per_second': 2.328}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'norm_preserving', 'completed_configurations': 3, 'seconds': 1.28, 'configurations_per_second': 2.335}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'gaussian_denoiser', 'completed_configurations': 3, 'seconds': 2.08, 'configurations_per_second': 1.441}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'sae_calibrated', 'completed_configurations': 3, 'seconds': 2.1, 'configurations_per_second': 1.431}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'fluency_denoiser', 'completed_configurations': 3, 'seconds': 2.11, 'configurations_per_second': 1.422}
{'stage': 'token_evaluation', 'prompt_batch': 0, 'method': 'projected_fluency_denoiser', 'completed_configurations': 3, 'seconds': 2.08, 'configurations_per_second': 1.444}
{'stage': 'token_evaluation'

## 24. Correction geometry


In [24]:
if neighbor_results is not None:
    geometry_rows = neighbor_results[neighbor_results['nearest_clean_correction_cosine'].notna()]
    geometry_rows.to_csv(analysis_dir/'correction_geometry.csv', index=False)
    print(geometry_rows.groupby('method')['nearest_clean_correction_cosine'].agg(['mean','median','std']))
else:
    print('Correction geometry skipped with neighbor diagnostics')


                                mean    median       std
method                                                  
fluency_denoiser            0.146442  0.137350  0.034232
gaussian_denoiser           0.071155  0.075426  0.009977
incremental_fluency         0.068609  0.042936  0.065838
projected_fluency_denoiser  0.001701  0.000175  0.002784
sae_calibrated              0.160742  0.148732  0.044548


## 25. Validation generation


In [25]:
generation_results = pd.DataFrame()
if RUN_GENERATION and not RUN_TEST:
    generation_methods = [method for method in config['evaluation']['generation_methods']
        if method in main_methods]
    generation_results = evaluate_generation_methods(model, sae,
        context_texts[:config['evaluation']['generation_num_prompts']], all_directions,
        direction_split, selected_val_ids, config['evaluation']['full_strengths'],
        config['evaluation']['generation_seeds'], hook_name, denoisers, normalizations,
        config['evaluation'], 'val', generation_methods,
        jsonl_path=config['evaluation']['generation_jsonl_path'],
        results_path=config['evaluation']['results_path'],
        aggregate_path=config['evaluation']['aggregate_results_path'],
        checkpoint_fingerprints=checkpoint_fingerprints, profiler=runtime_profile,
        sae_identity=SAE_IDENTITY)
    print(generation_results.groupby(['method','alpha'])[['clean_model_nll','kl','concept_score']].mean())
runtime_profile.save(analysis_dir/'runtime_profile.csv')


PosixPath('outputs/analysis/runtime_profile.csv')

## 26. Freeze configuration after validation


In [26]:
frozen_path = None
if not DEBUG and not RUN_TEST:
    assert Path(validation_dir/'main_token_metrics.csv').exists()
    frozen_path = freeze_test_config(config, main_methods)
    print('Frozen test config:', frozen_path)
elif DEBUG:
    print('DEBUG run does not freeze the final test configuration')
else:
    print('RUN_TEST=True: using the existing frozen configuration')


Frozen test config: outputs/frozen_test_config.json


## 27. Optional one-time final test


In [27]:
test_results = pd.DataFrame()
if RUN_TEST:
    frozen = load_frozen_test_config('outputs/frozen_test_config.json')
    validate_direction_ids_for_usage(test_ids, direction_split, 'final_evaluation', require_complete_split=True)
    test_results = evaluate_token_level_methods_fast(model, full_batches, all_directions,
        direction_split, test_ids, frozen['strengths'], hook_name, frozen['methods'],
        denoisers, normalizations, frozen['evaluation'], 'test', sae,
        intervention_batch_size=EVAL_INTERVENTION_BATCH,
        use_inference_autocast=USE_INFERENCE_AUTOCAST,
        partial_jsonl_path=Path('outputs/final_test/token_metrics.partial.jsonl'),
        checkpoint_fingerprints=checkpoint_fingerprints, sae_identity=SAE_IDENTITY,
        profiler=runtime_profile)
    final_test_dir = Path('outputs/final_test'); final_test_dir.mkdir(parents=True, exist_ok=True)
    test_results.to_csv(final_test_dir/'token_metrics.csv', index=False)
    test_results.groupby(['method','direction_id','strength'], as_index=False)[
        ['kl','clean_nll','modified_nll','delta_nll','activation_norm_ratio',
         'target_sae_activation']].mean().to_csv(
            final_test_dir/'token_level_aggregate.csv', index=False)
    if RUN_GENERATION:
        evaluate_generation_methods(model, sae,
            context_texts[:config['evaluation']['final_generation_num_prompts']],
            all_directions, direction_split, test_ids, frozen['strengths'],
            config['evaluation']['final_generation_seeds'], hook_name, denoisers, normalizations,
            frozen['evaluation'], 'test', config['evaluation']['generation_methods'],
            jsonl_path=final_test_dir/'generations.jsonl',
            results_path=final_test_dir/'generation_metrics.csv',
            aggregate_path=final_test_dir/'generation_aggregate.csv',
            checkpoint_fingerprints=checkpoint_fingerprints, profiler=runtime_profile,
            sae_identity=SAE_IDENTITY)
else:
    print('RUN_TEST=False: test directions remain untouched')


RUN_TEST=False: test directions remain untouched


## 28. Final plots and tables


In [28]:
plot_source = generation_results if not generation_results.empty else validation_results
figure_dir = Path('outputs/debug/final_figures' if DEBUG else 'outputs/final_figures')
created_figures = generate_standard_figures(plot_source, figure_dir,
    damage_scores=score_payload['mean_kl'], neighbor_diagnostics=neighbor_results)
print({name: str(path) for name, path in created_figures.items()})


{'pareto': 'outputs/final_figures/pareto_fluency_concept.png', 'concept_nll': 'outputs/final_figures/concept_vs_nll.png', 'alpha_delta_nll': 'outputs/final_figures/alpha_vs_delta_nll.png', 'alpha_norm': 'outputs/final_figures/alpha_vs_activation_norm_ratio.png', 'damage_histogram': 'outputs/final_figures/direction_damage_histogram.png', 'knn_delta_nll': 'outputs/final_figures/knn_distance_vs_delta_nll.png'}


## 29. Report files and output manifest


In [29]:
analysis_dir = Path('outputs/debug/analysis' if DEBUG else 'outputs/analysis')
analysis_dir.mkdir(parents=True, exist_ok=True)
manifest = {'pipeline_version': PIPELINE_VERSION, 'debug': DEBUG,
    'validation_rows': len(validation_results), 'generation_rows': len(generation_results),
    'figures': {key: str(value) for key, value in created_figures.items()},
    'checkpoints': {key: str(value) for key, value in paths.items() if Path(value).exists()}}
(analysis_dir/'manifest.json').write_text(json.dumps(manifest, indent=2)+'\n', encoding='utf-8')
print(json.dumps(manifest, indent=2))


{
  "pipeline_version": "final_v2",
  "debug": false,
  "validation_rows": 336,
  "generation_rows": 0,
  "figures": {
    "pareto": "outputs/final_figures/pareto_fluency_concept.png",
    "concept_nll": "outputs/final_figures/concept_vs_nll.png",
    "alpha_delta_nll": "outputs/final_figures/alpha_vs_delta_nll.png",
    "alpha_norm": "outputs/final_figures/alpha_vs_activation_norm_ratio.png",
    "damage_histogram": "outputs/final_figures/direction_damage_histogram.png",
    "knn_delta_nll": "outputs/final_figures/knn_distance_vs_delta_nll.png"
  },
  "checkpoints": {
    "gaussian": "outputs/checkpoints/gaussian.pt",
    "sae_calibrated": "outputs/checkpoints/sae_calibrated.pt",
    "fluency": "outputs/checkpoints/fluency.pt"
  }
}


## 30. Final factual summary and automated audit


In [30]:
audit_paths = {'gaussian': paths['gaussian'], 'sae_calibrated': paths['sae_calibrated'],
               'fluency': paths['fluency']}
audit = run_pipeline_audit(notebook_path='notebooks/final_v3_all_experiments.ipynb', model=model, sae=sae,
    hook_name=hook_name, direction_split=direction_split, smoke_results=validation_results,
    config=config, calibrated_sampler=calibrated_sampler, checkpoint_paths=audit_paths)
audit_path = Path('outputs/debug/audit.json' if DEBUG else 'outputs/analysis/audit.json')
audit_path.parent.mkdir(parents=True, exist_ok=True)
audit_path.write_text(json.dumps({'pipeline_version': PIPELINE_VERSION, 'checks': audit}, indent=2)+'\n')
runtime_profile.save(analysis_dir/'runtime_profile.csv')
runtime_profile.print_summary()
if torch.cuda.is_available():
    print({'cuda_memory_allocated_gib': torch.cuda.memory_allocated()/2**30,
           'cuda_max_memory_allocated_gib': torch.cuda.max_memory_allocated()/2**30})
print('Completed corrected pipeline. Results are empirical; no improvement is assumed.')


PIPELINE AUDIT
[OK] no legacy notebook execution
[OK] model/SAE dimensions match
[OK] split disjoint
[OK] no test leakage
[OK] normalization round trip
[OK] calibrated standalone sampler math
[OK] correction diagnostics invariant
[OK] evaluation uses inference mode
[OK] no multiprocessing DataLoader
[OK] no NaN/inf in smoke results
[OK] raw/raw concept retention
[OK] checkpoint schema compatible
[OK] calibrated REAL training batch math
                       stage   seconds  clean_forwards  modified_forwards  configurations  examples  configurations_per_second
      SAE feature extraction 27.667819               0                  0               0         0                   0.000000
               clean forward  0.358531              12                  0               0       112                   0.000000
      dataframe construction  0.032186               0                  0               0         0                   0.000000
   harmfulness clean forward  0.062464              

## 12–18. V3 architecture, calibrated objectives, training, validation selection, and freeze

`GatedConditionedDenoiser` receives `(x, v, s)` and returns
`x + g(s) C(x,v,s)`, with `g(0)=0`. The runner trains four pre-registered
ablations under identical architecture/optimizer settings. Auxiliary loss
scales come only from TRAIN batches. Validation chooses the frozen method;
holdout tensors are not tokenized before the protocol and checkpoint hashes
are written. Gaussian, harmfulness weighting, projection, and incremental
variants are supporting or negative controls rather than central proposals.


## 19–28. Frozen holdout, generation, semantic proxy, diagnostics, figures, and report

The following single call completes every remaining stage. `DEBUG_V3=False`
is the production setting. Change it to `True` only for a short path smoke test.
The independent classifier is explicitly secondary and is not human ground truth.


In [31]:
from run_v3 import run_all

# Production one-shot run. V1 artifacts were created above in this same kernel.
DEBUG_V3 = False
RUN_V3_HOLDOUT = True
RUN_V3_GENERATION = True
RUN_V3_SEMANTIC_PROXY = True

run_all(
    debug=DEBUG_V3,
    run_holdout=RUN_V3_HOLDOUT,
    run_generation=RUN_V3_GENERATION,
    run_semantic=RUN_V3_SEMANTIC_PROXY,
    preloaded_model=model,
    preloaded_sae=sae,
    preloaded_directions=all_directions,
)


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

conditioned_reconstruction {"step": 1.0, "loss": 1.1583054065704346, "weighted_kl": 0.0, "weighted_retention": 0.0, "weighted_reconstruction": 1.1583054065704346, "weighted_correction": 1.3664417395987016e-17, "retention_ratio": 1.0}
conditioned_reconstruction {"step": 25.0, "loss": 1.0151444673538208, "weighted_kl": 0.0, "weighted_retention": 0.0, "weighted_reconstruction": 1.0151411294937134, "weighted_correction": 3.3134513159893686e-06, "retention_ratio": 1.0040796995162964}
conditioned_reconstruction {"step": 50.0, "loss": 0.4808410704135895, "weighted_kl": 0.0, "weighted_retention": 0.0, "weighted_reconstruction": 0.4807918667793274, "weighted_correction": 4.921153959003277e-05, "retention_ratio": 1.035041332244873}
conditioned_reconstruction {"step": 75.0, "loss": 1.6329123973846436, "weighted_kl": 0.0, "weighted_retention": 0.0, "weighted_reconstruction": 1.632698893547058, "weighted_correction": 0.000213483304833062, "retention_ratio": 1.054781198501587}
conditioned_reconstruc

Generation eval (new_unseen_v3_holdout):   0%|          | 0/960 [00:00<?, ?it/s]

Moving model to device:  cpu


...

Loaded pretrained model gpt2-small into HookedTransformer


Semantic generations: 100%|██████████| 576/576 [10:29<00:00,  1.09s/it]


Moving model to device:  cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2352.51it/s]


{
  "pipeline_version": "semantic_eval_v1",
  "selected_direction": {
    "feature_id": 6516,
    "split_membership": "outside",
    "feature_6516_split_membership": "outside",
    "feature_6516_legally_usable": true,
    "selection_reason": "preselected public feature outside the canonical sampled split",
    "public_interpretation": "greeting-related words and phrases; greetings and welcoming interactions",
    "source_url": "https://www.neuronpedia.org/gpt2-small/6-res-jb/6516",
    "hypothesis": "This text is a greeting or welcoming interaction.",
    "control_hypothesis": "This text is unrelated to greetings or welcoming interactions.",
    "selected_before_classifier_evaluation": true,
    "negative_control_feature_id": 9696,
    "negative_control_split_membership": "outside",
    "negative_control_public_interpretation": "references to vacuums and vacuum-related language",
    "candidate_memberships": {
      "6516": "outside",
      "9696": "outside",
      "7672": "outside"
  

## 28. Artifact manifest

In [32]:
from pathlib import Path

final_root = Path('outputs/final_v3')
required_final_artifacts = [
    'results/token_validation.csv',
    'results/token_holdout_or_replication.csv',
    'results/ablation_summary.csv',
    'results/causal_ablation_table.csv',
    'results/harmfulness.csv',
    'results/correction_geometry.csv',
    'results/statistical_tests.csv',
    'results/holdout_statistical_tests.csv',
    'results/cross_concept_confirmation.csv',
    'results/cross_concept_statistics.csv',
    'generations/final_generation.jsonl',
    'configs/frozen_protocol.json',
    'diagnostics/loss_scale_diagnostic.json',
    'diagnostics/runtime_breakdown.json',
    'diagnostics/audit.json',
    'final_report.md',
    'final_summary.json',
]
missing = [name for name in required_final_artifacts if not (final_root / name).exists()]
assert not missing, f'Missing final V3 artifacts: {missing}'
for path in sorted(final_root.rglob('*')):
    if path.is_file():
        print(path, path.stat().st_size)


outputs/final_v3/checkpoints/conditioned_full.pt 53227141
outputs/final_v3/checkpoints/conditioned_kl.pt 53226965
outputs/final_v3/checkpoints/conditioned_kl_retention.pt 53227909
outputs/final_v3/checkpoints/conditioned_reconstruction.pt 53228085
outputs/final_v3/configs/cross_concept_direction.json 566
outputs/final_v3/configs/direction_split_v3.json 1881
outputs/final_v3/configs/frozen_protocol.json 6340
outputs/final_v3/configs/literal_alpha_calibration.json 361
outputs/final_v3/configs/prompt_split_manifest.json 437
outputs/final_v3/configs/sequence_text_bank.json 964667
outputs/final_v3/diagnostics/audit.json 759
outputs/final_v3/diagnostics/harmfulness_correlations.json 169
outputs/final_v3/diagnostics/loss_scale_diagnostic.json 1657
outputs/final_v3/diagnostics/neighbor_correlations.json 98
outputs/final_v3/diagnostics/numerical_gates.json 522
outputs/final_v3/diagnostics/runtime_breakdown.csv 752
outputs/final_v3/diagnostics/runtime_breakdown.json 2045
outputs/final_v3/diagnos